In [0]:
from pyspark.sql.functions import *

store_df = spark.read.table("fk_ce_2026_07_1382.bronze.stores")
standardized_orders_df = (
    store_df
    .drop("file_path", "ingestion_timestamp")
    .withColumns({
        "store_id": upper(trim(col("store_id"))),
        "store_name": trim(col("store_name")),
        "city": trim(col("city")),
        "state": trim(col("state")),
        "region": trim(col("region")),

        "opening_date": try_to_date(col("opening_date"), "yyyy-MM-dd")
    })
)

standardized_orders_df.write.mode("overwrite").saveAsTable("fk_ce_2026_07_1382.silver.stores")

In [0]:
from pyspark.sql.functions import *

df_products = spark.read.table("fk_ce_2026_07_1382.bronze.products")
standardized_products_df = (
    df_products
    .withColumns({
        "product_id": upper(trim(col("product_id"))),
        "product_name": trim(col("product_name")),
        "brand": trim(col("brand")),

        "category_l1": col("category.l1"),
        "category_l2": col("category.l2"),

        "pack_size": col("pack.size").cast("int"),
        "pack_unit": col("pack.unit"),

        "mrp": col("mrp").cast("decimal(10,2)"),
        "selling_price": col("selling_price").cast("decimal(10,2)"),

        "last_changed_snapshot_date": try_to_date(col("snapshot_date"), "yyyy-MM-dd")
    })
    .drop("file_path", "ingestion_timestamp", "category", "pack")
)

In [0]:
standardized_products_df.createOrReplaceTempView("standardized_products")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS fk_ce_2026_07_1382.silver.products (
  product_id STRING,
  product_name STRING,
  brand STRING,
  is_active BOOLEAN,
  mrp DECIMAL(10,2),
  selling_price DECIMAL(10,2),
  snapshot_date STRING,
  tags ARRAY<STRING>,
  category_l1 STRING,
  category_l2 STRING,
  pack_size INT,
  pack_unit STRING,
  last_changed_snapshot_date DATE
)

In [0]:
%sql
MERGE INTO fk_ce_2026_07_1382.silver.products AS t
USING standardized_products AS s
ON t.product_id = s.product_id
WHEN MATCHED THEN
  UPDATE SET * 
WHEN NOT MATCHED THEN
  INSERT *;

In [0]:
customer_df = spark.read.table("fk_ce_2026_07_1382.bronze.customers")

In [0]:
%sql
select * from fk_ce_2026_07_1382.bronze.orders

In [0]:
from pyspark.sql.functions import *

df_orders = spark.read.table("fk_ce_2026_07_1382.bronze.orders")
standardized_orders_df = (
    df_orders
    .withColumn("item", explode(col("items")))
    .withColumns({
        "order_id": upper(trim(col("order_id"))),
        "customer_id": upper(trim(col("customer_id"))),
        "store_id": upper(trim(col("store_id"))),
        "product_id": upper(trim(col("item.product_id"))),

        "order_status": when(upper(trim(col("order_status"))) == "CANCELED", lit("CANCELLED")).otherwise(upper(trim(col("order_status")))),
        
        "payment_method": upper(trim(col("payment.method"))),

        "order_ts": coalesce(
            try_to_timestamp(col("order_ts"), lit("yyyy-MM-dd HH:mm:ss")),
            try_to_timestamp(col("order_ts"), lit("MM/dd/yyyy'T'HH:mm:ss"))
        ),

        "updated_at": coalesce(
            try_to_timestamp(col("updated_at"), lit("yyyy-MM-dd HH:mm:ss"))
        ),

        "quantity": expr("try_cast(item.qty as int)"), 

        "discount_amount": col("item.discount")
    })
    .filter(col("order_status").isin(["PLACED", "DELIVERED", "CANCELLED"]))
    .filter((col("quantity") > 0) & (col("quantity") % 1 == 0))
    .filter(col("item.unit_price") > 0)

    .fillna({"discount_amount": 0})
)


In [0]:
standardized_orders_df.createOrReplaceTempView("standardized_orders_df")

In [0]:
%sql
CREATE OR REPLACE TABLE fk_ce_2026_07_1382.silver.deduplicated_orders AS
WITH RankedOrders AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY order_id 
            ORDER BY updated_at DESC
        ) AS rn
    FROM standardized_orders_df
)
SELECT 
    *
FROM RankedOrders
WHERE rn = 1;

In [0]:
%sql
select * from fk_ce_2026_07_1382.silver.deduplicated_orders

from the above fk_ce_2026_07_1382.silver.deduplicated_orders table I want to create two table one is fk_ce_2026_07_1382.silver.orders which contain the orders which follow the rules it contains columns order_id, customer_id, store_id, order_ts, order_date, order_status, payment_method, promised_minutes, actual_minutes, delivery_fee and updated_at and fk_ce_2026_07_1382.silver.quarantine_orders which contains the rejected orders the columns are order_id, rejection_region, updated_at, raw_records, quarantined_at. The rules are missing_customer: ustomer_id is null, unknown customer: customer_id does not exists in silver_customer, unknown_store: store_id is null or does not exists in silver.stores, invalid timestamp order_ts is invalid or contains null, empty order: items is missing, product is missing: product_id is does not exists in silver.products , invalid quantity: when quantity is below 0 and not whole number, invalid price: when price is less than 0